- Example llma index RAG and PageIndex, let's convert to databricks context

In [0]:
# RAG — Standard Chunking:

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

# Load and chunk
documents = SimpleDirectoryReader('data/').load_data()
parser = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = parser.get_nodes_from_documents(documents)

# Build index — nodes are arbitrary token windows
index = VectorStoreIndex(nodes)
query_engine = index.as_query_engine(similarity_top_k=5)

# Gets 5 arbitrary chunks — context may be incomplete
response = query_engine.query('What are the termination rights?')

In [0]:
# PageIndex — Structure-Aware Retrieval:

from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SimpleFileNodeParser

# Parse with page/section awareness
parser = SimpleFileNodeParser()
nodes = parser.get_nodes_from_documents(documents)

# Nodes now carry: page_number, section_title, doc_name
for node in nodes[:2]:
    print(node.metadata)  # {'page_label': '3', 'section': 'Termination'...}

# Build index with richer metadata
index = VectorStoreIndex(nodes)
query_engine = index.as_query_engine(similarity_top_k=3)

# Response includes traceable page-level context
response = query_engine.query('What are the termination rights?')
print(response.source_nodes[0].metadata['page_label'])  # '14'

- important :: A note on the “token efficiency” tradeoff: RAG wins on token efficiency because smaller chunks mean you send less to the LLM per query. But this is a false economy if those chunks don’t contain the full context needed to answer correctly.